# Bước 00: Hotfix Căn Chỉnh Thời Tiết Causal (Re-align Weather Hotfix)
Dự án: Tốt nghiệp - Energy Forecasting - Nhóm thực hiện: The Outliers

## 1. TỔNG QUAN VÀ MỤC TIÊU
Notebook này đọc trực tiếp file `data/mlmart_base/v3_preprocessing.parquet`, loại bỏ 100% rò rỉ thời tiết tương lai bằng phép gán Causal Floor (`floor(timestamp, 1 hour)`), và ghi đè an toàn lại file gốc.

**Input:** `../../data/mlmart_base/v3_preprocessing.parquet`
**Output:** `../../data/mlmart_base/v3_preprocessing.parquet`


In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

INPUT_PATH = Path("../../data/mlmart_base/v3_preprocessing.parquet")
OUTPUT_PATH = Path("../../data/mlmart_base/v3_preprocessing.parquet")

WEATHER_COLUMNS = (
    'weather_id', 'weather_type_id', 'weather_timestamp', 'weather_is_day',
    'shortwave_radiation', 'direct_normal_irradiance', 'diffuse_solar_radiation',
    'temperature_c', 'cloud_cover_total', 'cloud_cover_low', 'cloud_cover_mid',
    'cloud_cover_high', 'wind_speed', 'precipitation_mm', 'sunshine_duration',
    'weather_code', 'weather_type_is_day', 'weather_condition', 'weather_description'
)
LOOKUP_KEY = ("site_id", "_weather_hour")

print("Đã import thư viện và khai báo tham số.")
print(f"- Input / Output: {INPUT_PATH}")


In [ ]:
# ── 1. Đọc dữ liệu và đo lường số dòng rò rỉ tương lai TRƯỚC khi sửa ──
df = pd.read_parquet(INPUT_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['weather_timestamp'] = pd.to_datetime(df['weather_timestamp'])

_delta_truoc = (df['weather_timestamp'] - df['timestamp']).dt.total_seconds() / 60
_leak_truoc = int((_delta_truoc > 0).sum())
print(f"Tổng số dòng: {len(df):,}")
print(f"Dòng dùng thời tiết TƯƠNG LAI trước khi sửa: {_leak_truoc:,}/{len(df):,} ({_leak_truoc / len(df) * 100:.2f}%)")


In [ ]:
# ── 2. Thực hiện căn chỉnh thời tiết theo quy tắc Causal Floor ──
_tt = [c for c in WEATHER_COLUMNS if c in df.columns]

_bang_tt = (
    df[['site_id', 'weather_timestamp'] + _tt]
    .dropna(subset=['weather_timestamp'])
    .drop_duplicates(['site_id', 'weather_timestamp'])
    .rename(columns={'weather_timestamp': '_nhan_gio'})
)

df['_nhan_gio'] = df['timestamp'].dt.floor('h')
df = df.drop(columns=_tt).merge(_bang_tt, on=['site_id', '_nhan_gio'], how='left')
df['weather_timestamp'] = df['_nhan_gio']
df = df.drop(columns=['_nhan_gio'])

print("Đã căn chỉnh lại 19 cột thời tiết theo quy tắc Causal Floor.")


In [ ]:
# ── 3. Kiểm tra kết quả SAU khi sửa và Cổng kiểm soát (Assertion) ──
_delta_sau = (df['weather_timestamp'] - df['timestamp']).dt.total_seconds() / 60
_leak_sau = int((_delta_sau > 0).sum())
print(f"Dòng dùng thời tiết TƯƠNG LAI sau khi sửa: {_leak_sau:,} (Phải bằng 0)")

assert _leak_sau == 0, "LỖI BẢO VỆ: Vẫn còn dòng rò rỉ thời tiết tương lai!"
print("CỔNG KIỂM TRÁ: ĐẠT — 100% Causal Non-leaking Weather Data!")


In [ ]:
# ── 4. Ghi đè file đĩa an toàn ──
df.to_parquet(OUTPUT_PATH, index=False)
print(f"HOÀN TẤT: Đã ghi đè file dữ liệu hotfix tại: {OUTPUT_PATH}")
